# Benchmark DeepGaze MSDB on all of UEyes, on a free Colab GPU

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

One-time setup: put `ueyes_slim.zip` (~0.9 GB: UEyes `images/`, `eyetracker_logs/`, `image_types.csv`; Jiang et al., CHI 2023, CC BY 4.0) in the top level of My Drive.

Per-image results are saved to `My Drive/MediaAttention-bench/`, so if the session drops, reconnect and Run all again: it continues where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MA_BENCH'] = '/content/bench'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -e
D=/content/bench/data/ueyes
mkdir -p $D /content/drive/MyDrive/MediaAttention-bench/cache
ln -sfn /content/drive/MyDrive/MediaAttention-bench/cache /content/bench/cache
if [ ! -f $D/UEyes_dataset/image_types.csv ]; then cp /content/drive/MyDrive/ueyes_slim.zip /content/ && unzip -q /content/ueyes_slim.zip -d $D && rm /content/ueyes_slim.zip; fi
ls $D/UEyes_dataset/images | wc -l

In [ ]:
!git clone -q https://github.com/pranikz/MediaAttention /content/MediaAttention 2>/dev/null || git -C /content/MediaAttention pull -q
!pip install -q -e "/content/MediaAttention[unisal]" "git+https://github.com/matthias-k/pysaliency.git@dev" pandas scikit-image einops clip-anytorch fire "deepgaze_pytorch @ git+https://github.com/matthias-k/DeepGaze"

In [ ]:
%cd /content/MediaAttention
!python benchmarks/run.py ueyes --models deepgaze_msdb 2>&1 | tr '\r' '\n' | awk 'NR % 50 == 0 || /done|Error|Traceback/'

In [ ]:
import json
print(json.dumps(json.load(open('benchmarks/results/ueyes.json'))['deepgaze_msdb'], indent=1))
!cp benchmarks/results/ueyes.json benchmarks/results/ueyes_per_image_deepgaze_msdb.csv /content/drive/MyDrive/MediaAttention-bench/ && ls -la /content/drive/MyDrive/MediaAttention-bench/